# Reliable ML Evaluation: Baselines + Cross-Validation

## 1. Retrieval

1. Why did we split Iris into X_train/y_train and X_test/y_test before training Gaussian Naive Bayes?

> Because the data we use to test the model cannot be part of the same dataset that was used to train the model, otherwise the test data will have already been used to set the model parameters and the test becomes invalid.

2. What would be wrong with estimating the Gaussian means and variances using both X_train and X_test?

> Exposing the model to the data that will be used to test it during training would cause data leakage as the model should not be exposed to information during training that wouldn't be available at the time of prediction. This invalidates the test results as it can lead to artificially high scores on benchmarks that use this test data.

3. What did stratify=y do when we called train_test_split()?

> It ensures that the training and testing sets have the same proportion of classes as the original dataset.

4. Write the formula for classification accuracy.

> $\text{accuracy}=\frac{\text{Number of correct predictions made}}{\text{Number of predictions made}}$

5. In Gaussian Naive Bayes, what information was used to estimate the class prior $P(C_k)$?

> The prior is estimated from the proportion of `training observations` belonging to each class: 
$$P(C_k)\approx\frac{\text{number of training observations in }C_k}{\text{total training observations}}$$


## 2. Baselines

Suppose a classifier gets 70% accuracy. Is that good?

Not necessarily.

Imagine 70% of the observations belong to class 0. A "model" that completley ignores the features and simply predicts:

> class 0 every single time

would also achieve 70% accuracy.

So before asking whether a real model performs well, we need a simple reference point.

A **baseline model** is a deliberatley simple prediction strategy that gives us a minimum standard our real model should beat.

For classification, one useful baseline is:

> Always predict the most common class in the training data.

Skikit-learn provides `DummyClassifier` for exactly this sort of comparison. It doesn't learn meaningful relationships between $X$ and $y$; its purpose is to tell us what performance we can achieve **without using the predictive information in the features**.

So if: $$\text{Dummy accuracy}=0.63$$
and: $$\text{GaussianNB accuracy}=0.94$$
then the interesting statement isn't mereley "GaussianNB gets 94%."

It's:

> GaussianNB substantially outperforms a simple baseline, suggesting that the features contains useful predictive information.

The baseline doesn't tell us the model is *good enough for deployment*. It simply gives us context. 

## 3. Cross-validation

Our Iris workflow used one train/test split: $$\text{training data}\quad\mid\quad\text{test data}$$

We trained on the first part and evaulated on the second.

That's suitable for obtaining a final unbiased test result, but there's a problem if we want to **choose between models**.

Suppose we have:
- GaussianNB
- Model B
- Model C

If we repeatedly evaluate all three on the test set and choose whichever performs best there, then the test set has influenced our model choice.

Even though we haven't literally fitted model parameters on the test observations, we have indirectly adapted our decisions to them. The test set is no longer a clean final evaluation.

### We therefore need validation data

Conceptually, we'd like: $$\text{training data}\quad\mid\quad\text{validation data}\quad\mid\quad\text{test data}$$

The roles are different:
- **training**: fit model parameters;
- **validation**: compare models / make modelling decisions;
- **test**: evaluate the chosen approach once at the end.

But if the dataset isn't huge, permanently setting aside a separate validation set wastes useful training data.

That's where **cross-validation** comes in.

### 5-fold cross-validation

Take the training data and divide it into five roughly equal folds: $$F_1,F_2,F_3,F_4,F_5$$

Run five experiments.

For the first: $$\text{fit on }F_2,F_3,F_4,F_5\quad\rarr\quad\text{validate on }F_1$$

Then: $$\text{fit on }F_1,F_3,F_4,F_5\quad\rarr\quad\text{validate on }F_2$$

and continue until every fold has served as the validation fold exactly once. 

You therefore obtain five validation scores: $$a_1,a_2,a_3,a_4,a_5 $$

Rather thn juding a model from one particular split we can report: 

$$\text{mean CV accuracy }=\frac{1}{5}\sum_{i=1}^5a_i$$

and the spread of those scores, commonly their stadard deviation.

The mean tells us the model's typical validation performance.

The spread gives us some indication of how sensitive that performance is to which observations happened to be in the validation fold,

### The crucial structure

Throughout all five folds, the real test set remains completely untouched:

$$\boxed{\text{training set}\rarr\text{cross-validation/model selection}\rarr\text{choose model}\rarr\text{test set once}}$$

## 4. Conceptual check

1. Suppose GuassianNB gets CV accuracies: $$[0.91, 0.94. 0.92, 0.95, 0.93]$$ while a dummy baseline gets: $$[0.63, 0.63, 0.62, 0.64, 0.63]$$ What useful information does the dummy model give us?

> The dummy model tells us how well could we perform without learning any useful relationship between the features and the target. In this example, since the GuassianNB model gets about $93%$ accuracy while the dummy gets around $63%$ accuracy, we can see that the GaussianNB is doing substantially better than a trivial strategy.

2. Why would the following be bad practise: Run GaussianNB, logistic regression and random forest on the **test set**, then select whichever has the highest test accuracy.

> In this scenario, the test set has influenced which model you choose. This means the final test result is no longer an independent estimate of how your chosen modelling process performs on unseen data.

3. During 5-fold cross-validation, does each observation in the training set get used:
    - for fitting?

    > Each training observation is used for fitting four times across five CV runs.
 
    - for validation?

    > Each training observation is used for validation exactly once across five CV runs

## 5. Main coding exercise — baseline + cross-validation

This is the single main implementation task for the session.

Use scikit-learn’s Breast Cancer Wisconsin dataset.

Your workflow should be:

$$ \text{load data} \rightarrow \text{hold out test set} \rightarrow \text{CV on training set} \rightarrow \text{select model} \rightarrow \text{evaluate selected model once on test set} $$

### Requirements
1. Load the Breast Cancer Wisconsin dataset.
2. Create an 80/20 train/test split using:
    - `random_state=42`
    - stratification by the target.
3. Report the class counts/proportions in:
    - the full dataset;
    - training set;
    - test set.
4. Create two candidate models:
    - a `DummyClassifier` using the most-frequent-class strategy;
    - `GaussianNB`.
5. Run 5-fold cross-validation using the training data only for each model, using accuracy.
6. For each model report:
    - all five CV accuracies;
    - mean CV accuracy;
    - standard deviation of CV accuracy.
7. Based only on the CV results, state which model you would select.
8. Fit that selected model to the whole training set.
9. Evaluate it once on the untouched test set and report test accuracy.
10. Write 2–3 sentences interpreting the result, including what the dummy baseline tells you.

In [ ]:
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score
from sklearn.dummy import DummyClassifier
from sklearn.naive_bayes import GaussianNB

# Load Breast Cancer Wisconsin dataset
breast_cancer = load_breast_cancer()

X = breast_cancer.data
y = breast_cancer.target

# Create an 80/20 train/test split
X_train, X_test, y_train, y_test = train_test_split(
  X,
  y,
  test_size=0.2,
  random_state=42,
  stratify=y 
)

# Report class priors
def calc_priors(y):
  classes = np.unique(y)
  priors = {}

  for c in classes:
    prior = np.count_nonzero(y == c) / len(y)
    priors[c] = prior

  return priors

full_priors = calc_priors(y)
train_priors = calc_priors(y_train)
test_priors = calc_priors(y_test)

print('Full dataset priors:', full_priors)
print('Training dataset priors:', train_priors)
print('Test dataset priors:', test_priors)

# Create two candidate models:
dummy_clf = DummyClassifier(strategy='most_frequent')
gnb = GaussianNB()

# Run 5-fold cross-validation on both models
dummy_CV_scores = cross_val_score(dummy_clf, X_train, y_train, cv=5)
gnb_CV_scores = cross_val_score(gnb, X_train, y_train, cv=5)

# Cross validation reporting
labels = ['Dummy CV', 'GaussianNB CV']

for index, scores in enumerate([dummy_CV_scores, gnb_CV_scores]):
  print(f'\n{labels[index]} scores: {scores}')
  print(f'{labels[index]} mean: {np.mean(scores): .3f}')
  print(f'{labels[index]} std: {np.std(scores): .3f}')

# Choose a model based on CV results
print(
  '\nBased on CV results, gnb performs substantially better than the dummy classifier on the training data (93% versus 62% accuracy)\nSelecting GaussianNB for whole training set fitting'
)
gnb.fit(X_train, y_train)

# Evaluate once on the test set
gnb_y_pred = gnb.predict(X_test)

n_correct_predictions = np.count_nonzero(gnb_y_pred == y_test)
accuracy = n_correct_predictions / len(gnb_y_pred)
print(f'\nFinal test accuracy: {accuracy: .3f}')


Full dataset priors: {np.int64(0): np.float64(0.37258347978910367), np.int64(1): np.float64(0.6274165202108963)}
Training dataset priors: {np.int64(0): np.float64(0.37362637362637363), np.int64(1): np.float64(0.6263736263736264)}
Test dataset priors: {np.int64(0): np.float64(0.3684210526315789), np.int64(1): np.float64(0.631578947368421)}

Dummy CV scores: [0.62637363 0.62637363 0.62637363 0.62637363 0.62637363]
Dummy CV mean:  0.626
Dummy CV std:  0.000

GaussianNB CV scores: [0.93406593 0.94505495 0.93406593 0.93406593 0.93406593]
GaussianNB CV mean:  0.936
GaussianNB CV std:  0.004

Based on CV results, gnb performs substantially better than the dummy classifier on the training data (93% versus 62% accuracy)
Selecting GaussianNB for whole training set fitting

Final test accuracy:  0.939


> The final test accuracy (0.939) is very close to the GaussianNB mean CV accuracy (0.936). The low CV standard deviation (0.004) indicates that validation performance was consistent across the five folds, while the similar test result provides additional evidence that the CV estimate generalised well to the held-out test set. We can also see that the Dummy CV scores match the highest class prior of the training dataset exactly (approx 0.626), which confirms that the dummy model is classifying all observations as the most frequent class. Overall, the GaussianNB model performs substantially better than the dummy model.